# 第63章 分面图（FacetGrid）

<!-- module-learning-arc:start -->
> **Seaborn 模块主线｜第 20 / 20 步：组织多变量、矩阵和分面证据**
>
> **持续应用背景：** 开展客群消费行为差异研究：先固定样本和统计语义，再比较分布、关系和分面结果，判断差异是否稳定。
>
> **承接上一阶段：** 聚类热力图（clustermap）  →  **本章任务：** 分面图（FacetGrid）  →  **下一步：** 模块大作业《客群消费行为差异研究》
>
> **大作业连接：** 本章练习将成为《客群消费行为差异研究》的一部分，最终需要从样本口径和分布比较走到关系验证、分面研究与因果边界说明。
<!-- module-learning-arc:end -->


## 本章场景

当数据里同时存在好几类对象（比如不同渠道的订单）时，把它们的散点挤在同一张图上常会糊成一团，看不出各自的趋势。


## 本章目标

学完本章，你将能够：

- **理解**：理解「分面图（FacetGrid）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「分面图（FacetGrid）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「分面图（FacetGrid）」并读出其中的结论。


## 适用场景

**背景引入**：当数据里同时存在好几类对象（比如不同渠道的订单）时，把它们的散点挤在同一张图上常会糊成一团，看不出各自的趋势。分面图会把每个类别单独放进一个小图里，按行或列整整齐齐排开，让你一眼就能横向对比“哪个渠道卖得更多、曲线关系是否更强”。这种“小倍数”布局在报告和仪表盘里很常见，学会用它，拥挤的图会立刻清晰不少。

一个图过于拥挤，需要按行列分组重复相同图形。


## 数据结构

长表，包含X、Y以及一至两个分面分类变量。


## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 col_wrap=3 改为 col_wrap=2，观察分面换行对布局的影响
2. 修改 sharex=True 为 sharex=False，对比共享与独立X轴对子图比较的作用
3. 调整 aspect 参数（如 0.8 或 1.2），说明子图宽高比对可读性的影响


## 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `sns.relplot()`、`grid.set_axis_labels()`、`grid.set_titles()`、`grid.fig.suptitle()` | 一个图过于拥挤，需要按行列分组重复相同图形。 | 面板过多 |
| 进阶变体 | `sns.catplot()`、`grid.set_axis_labels()`、`grid.set_titles()`、`grid.fig.suptitle()` | 在基础图表上增加分组、注释、布局或交互 | 坐标不共享却直接比较高低 |
| 关键参数 | `row/col` | 分面 | 面板过多 |
| 关键参数 | `col_wrap` | 换行 | 坐标不共享却直接比较高低 |
| 关键参数 | `sharex/sharey` | 共享轴 | 分面和hue重复编码同一变量 |
| 关键参数 | `height/aspect` | 尺寸 | 面板过多 |


## 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 中文字体支持：由平台运行时自动配置
# 说明：Matplotlib 默认字体不含中文字形，中文会显示成方框。
#      本平台在运行每个绘图 cell 前，会自动注册可用的中文字体并设置
#      font.sans-serif / axes.unicode_minus，即使 seaborn 的 sns.set_theme
#      会重置字体，运行时也会在 set_theme 之后自动恢复。因此这里无需手动
#      import 或 addfont，直接使用即可。

# 1️⃣ 主题与数据导入：统一画风，读取三个公开数据集
sns.set_theme(style="whitegrid", context="notebook")

diamonds = pd.read_csv("/datasets/diamonds.csv")
taxis = pd.read_csv("/datasets/taxis.csv", parse_dates=["pickup", "dropoff"])
flights = pd.read_csv("/datasets/flights.csv")
print(
    f'Diamonds {len(diamonds):,} | Taxis {len(taxis):,} | Flights {len(flights):,} 行'
)


In [ ]:
# 2️⃣ 特征工程：把原始字段映射成图表统一使用的列名与派生指标
orders_full = diamonds.assign(
    category=diamonds["cut"],
    channel=diamonds["color"],
    region=diamonds["clarity"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    satisfied=np.where(
        diamonds["price"] >= diamonds["price"].median(),
        "高于中位价",
        "不高于中位价",
    ),
)
orders = orders_full.sample(2_000, random_state=36)

marketing_full = taxis.assign(
    channel=taxis["payment"].fillna("unknown"),
    visits=taxis["distance"],
    ad_spend=taxis["tip"],
    sales=taxis["total"],
    conversion=(taxis["tip"] / taxis["total"].replace(0, np.nan)).fillna(0),
)
marketing = marketing_full.sample(
    min(2_000, len(marketing_full)), random_state=36
).copy()

daily = flights.assign(
    date=pd.to_datetime(
        flights["year"].astype(str) + "-" + flights["month"] + "-01"
    ),
    region="AirPassengers",
    sales=flights["passengers"],
)
print(
    f'样本：orders {len(orders):,} | marketing {len(marketing):,} | daily {len(daily):,} 行'
)


## 例 1｜最小可用图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
import matplotlib.pyplot as plt

grid = sns.relplot(
    data=marketing,
    x="visits",
    y="sales",
    col="channel",
    col_wrap=3,
    hue="channel",
    height=3.3,
    aspect=1,
    palette="colorblind",
    legend=False,
)
grid.set_axis_labels("访问量", "销售额")
grid.set_titles("{col_name}")
grid.fig.suptitle("分渠道访问量与销售额", y=1.04)
plt.show()


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**：修改一个图表参数，观察分面布局的变化。

57.4 的基础分面图把不同渠道并排排成了 3 列。请在此基础上再做一次小改动：

1. 复制基础图表的绘图代码，把 `col_wrap` 从 `3` 改成 `2`，观察分面从“每行 3 个”变成“每行 2 个”时，总行数和每个子图的大小如何变化；
2. 若某个渠道的样本特别少、单看一个小图很费力，通常可以换一个分组更稳的分类变量来分面。先用一句话写下你的预期：把分面字段由样本稀少的类别换成样本较均衡的类别，会对可读性产生什么影响（无需真的改数据）。

提示：当前环境可能不弹出图像，可在下方单元格里用 `grid2.axes`、`marketing["channel"].nunique()` 核对子图数量来辅助判断。


In [ ]:
try:
    # 请在下方填写代码
    import matplotlib.pyplot as plt

    # TODO: 把下面这个开关改成 True，表示你已经复制了基础图表并修改了 col_wrap。
    done_figure = "待填写"  # <-- 请改成 True

    if done_figure is True:
        grid2 = sns.relplot(
            data=marketing,
            x="visits",
            y="sales",
            col="channel",
            col_wrap=2,  # TODO: 从 3 改成 2，观察分面换行
            hue="channel",
            height=3.3,
            aspect=1,
            palette="colorblind",
            legend=False,
        )
        grid2.set_axis_labels("访问量", "销售额")
        grid2.set_titles("{col_name}")
        grid2.fig.suptitle("分渠道访问量与销售额（col_wrap=2）", y=1.04)
        plt.show()

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
import matplotlib.pyplot as plt

grid = sns.catplot(
    data=orders,
    x="category",
    y="order_value",
    col="region",
    kind="box",
    hue="category",
    palette="Set2",
    legend=False,
    height=3.5,
    aspect=0.9,
)
grid.set_axis_labels("品类", "客单价（元）")
grid.set_titles("{col_name}")
grid.fig.suptitle("分区域品类客单价", y=1.04)
plt.show()


## 参数说明

- row/col：分面
- col_wrap：换行
- sharex/sharey：共享轴
- height/aspect：尺寸


## 结果解读

在共享坐标下比较模式、斜率和分布；同时检查每个面板样本量。


## 本章实训：分组比较与不确定性

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="region", y="sales", ci=None, ax=ax, color="#0F766E"
)
ax.set_title("地区销售额比较")
ax.set_ylabel("销售额")
plt.show()


### 第一个结果怎么读

Seaborn 负责把 DataFrame 的字段映射为图形编码；先明确横轴、纵轴和每行数据的粒度，再选择图表。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
report = report.sort_values("sales", ascending=False)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="sales", y="region", ci=None, ax=ax, color="#F59E0B"
)
ax.set_title("按销售额排序的地区比较")
ax.set_xlabel("销售额")
ax.set_ylabel("地区")
plt.show()


### 第二个结果怎么读

第二个实验只改变排序和坐标方向，让读者更容易找到最大值。图表调整必须服务于阅读任务。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：分组字段缺失怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
required = {"region", "sales"}
missing = required - set(report.columns)
if missing:
    print("缺少字段：", sorted(missing))
else:
    fig, ax = plt.subplots(figsize=(6, 3))
    sns.barplot(data=report, x="region", y="sales", ci=None, ax=ax)
    ax.set_title("地区销售额")
    plt.show()


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

绘图前先检查字段是否存在。把字段检查放在画图之前，错误会更接近真正原因，也更容易恢复。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 面板过多
- 坐标不共享却直接比较高低
- 分面和hue重复编码同一变量


## 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 独立迁移练习

修改一个分组、排序或统计设置，并比较修改前后的结论。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # 独立迁移练习：把 col 换成 row，观察行列分面的布局差异
    # 【目标】facet 的 col/row 决定小图排成横向还是纵向，练习改变布局。
    import matplotlib.pyplot as plt
    import seaborn as sns

    # 起点示例(已可运行)：col 换 row，让渠道纵向排列。
    grid = sns.relplot(
        data=marketing,
        x="visits",
        y="sales",
        row="channel",
        hue="channel",
        height=3,
        aspect=1.6,
        palette="colorblind",
        legend=False,
    )
    grid.set_axis_labels("访问量", "销售额")
    grid.fig.suptitle("分渠道访问量与销售额（行分面）", y=1.04)
    plt.show()

    # ---- 反思记录：横排 vs 纵排，比较单图尺寸和阅读顺序 ----
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(f"改动：{change_note}")
    print(f"预期：{expected_change}")
    print(f"观察：{observed_change}")

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 小结

使用FacetGrid、catplot和relplot把类别映射为可比较的小图。


### 你已经掌握

- 判断分面图（FacetGrid）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `row/col` | 分面 |
| `col_wrap` | 换行 |
| `sharex/sharey` | 共享轴 |
| `height/aspect` | 尺寸 |


### 需要注意

- 面板过多
- 坐标不共享却直接比较高低
- 分面和hue重复编码同一变量


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
# ===== 完整答案 =====
import matplotlib.pyplot as plt

# 1. 把 col_wrap 从 3 改为 2：分面由“每行 3 个”变成“每行 2 个”，
#    总行数变多、每个子图更扁更宽，方便在类别多时逐行逐列对比。
grid2 = sns.relplot(
    data=marketing,
    x="visits",
    y="sales",
    col="channel",
    col_wrap=2,
    hue="channel",
    height=3.3,
    aspect=1,
    palette="colorblind",
    legend=False,
)
grid2.set_axis_labels("访问量", "销售额")
grid2.set_titles("{col_name}")
grid2.fig.suptitle("分渠道访问量与销售额（col_wrap=2）", y=1.04)
plt.show()

# 2. 文字预期：改成样本较均衡的分类字段做分面后，每个子图样本量更充足、
#    组间对比更干净，比“每渠道一张小图”更容易读出差异；代价是丢掉了细分组的信息。


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
import matplotlib.pyplot as plt

facet = sns.FacetGrid(
    daily,
    row="region",
    height=2.2,
    aspect=3,
    sharex=True,
    sharey=True,
    margin_titles=True,
)
facet.map_dataframe(
    sns.lineplot, x="date", y="sales", marker="o", ci=None, color="#1a73e8"
)
facet.set_axis_labels("日期", "销售额")
facet.set_titles(row_template="{row_name}")
facet.fig.subplots_adjust(top=0.9)
facet.fig.suptitle("区域每日销售趋势")
plt.show()
